# Lab 2: Minimum Vertex Cover and Graph Coloring, Classical Methods and QUBO

**Course:** Quantum Optimization Lab
**Topics covered:** Minimum Vertex Cover, Graph Coloring
**Tools:** dimod, dwave-neal, networkx, numpy, matplotlib
**Session length:** 45 minutes, live walkthrough with in-class checkpoints

## Learning objectives

By the end of this lab you should be able to:

1. State the Minimum Vertex Cover and Graph Coloring problems and give at least one real industrial use case for each.
2. Solve small instances of both problems by hand, and check your answer against exact and heuristic classical solvers.
3. Explain, with a concrete measurement, where the classical approaches for each problem start to struggle.
4. Write the QUBO formulation for both problems and solve them with simulated annealing.
5. Compare classical and QUBO approaches on the same footing using a summary table.

This lab builds directly on Lab 1. If a term like QUBO, brute force, or simulated annealing is unfamiliar, go back to Lab 1 first.


In [ ]:
!pip install dimod dwave-neal networkx pulp qiskit qiskit-aer qiskit-optimization qiskit-algorithms pennylane -q

In [ ]:
import itertools
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import dimod
import neal

RNG_SEED = 7
rng = np.random.default_rng(RNG_SEED)
sampler = neal.SimulatedAnnealingSampler()

print("dimod version:", dimod.__version__)
print("neal version:", neal.__version__)


---
# Part 1: Minimum Vertex Cover

Given a graph $G = (V, E)$, a vertex cover is a subset of vertices $S \subseteq V$ such that every edge in the graph has at least one endpoint in $S$. Minimum Vertex Cover (MVC) asks for the smallest such set.

Put another way: if you had to place a guard on some of the vertices so that every edge is watched from at least one end, what is the fewest guards you would need, and where would you place them?


## 1.1 Why anyone outside a classroom cares about this

Minimum Vertex Cover shows up whenever "cover every connection with the fewest watchpoints" is the actual business problem:

- **Network monitoring.** Placing traffic sensors or firewalls on routers so that every network link is observed by at least one monitored router, using as few monitored routers as possible.
- **Wireless sensor networks.** Choosing a minimal set of sensor nodes to activate so that every communication link in the network is covered, to save battery life across the whole deployment.
- **Bioinformatics.** In protein interaction networks, a minimum vertex cover can identify a small set of proteins that, between them, touch every interaction in the network, which is useful for picking a minimal experimental panel.
- **VLSI and circuit testing.** Choosing the fewest test points on a circuit so that every connection between components can be checked from at least one test point.
- **Crew and resource scheduling.** If edges represent conflicts (two shifts that cannot both go unsupervised at the same time, say), a vertex cover finds the fewest supervisors needed so every conflicting pair has at least one supervisor present.

The common thread: edges are the things that must be "handled," vertices are the resources you place, and you want the fewest resources.


## 1.2 Three graphs you can solve by hand

Before touching any code, look at each graph below and try to find the minimum vertex cover yourself. Call it out loud, then run the reveal cell.

A useful trick while doing this by hand: every edge needs at least one endpoint covered, so start with the vertex touching the most edges, and see how far that gets you.


### Example A: a triangle (3 mutually connected vertices)

In [ ]:
GA = nx.complete_graph(3)
pos_a = nx.circular_layout(GA)
plt.figure(figsize=(3.5, 3.5))
nx.draw(GA, pos_a, with_labels=True, node_color="#1f4e79", font_color="white", node_size=600)
plt.title("Example A: triangle")
plt.show()


### Example B: a star graph (one hub, four leaves)

In [ ]:
GB = nx.star_graph(4)
pos_b = nx.spring_layout(GB, seed=RNG_SEED)
plt.figure(figsize=(4, 4))
nx.draw(GB, pos_b, with_labels=True, node_color="#e0a458", node_size=600)
plt.title("Example B: star graph")
plt.show()


### Example C: a path of 5 vertices

In [ ]:
GC_ = nx.path_graph(5)
pos_c = nx.spring_layout(GC_, seed=RNG_SEED)
plt.figure(figsize=(5, 2.5))
nx.draw(GC_, pos_c, with_labels=True, node_color="#4c8c4a", font_color="white", node_size=600)
plt.title("Example C: path graph, 5 vertices")
plt.show()


## 1.3 Classical approaches

We will look at two classical approaches: brute force, which is exact but exponential, and a matching based heuristic, which is fast and comes with a provable worst case guarantee.


### 1.3.1 Brute force

For $n$ vertices there are $2^n$ subsets to check. Brute force tries every subset, keeps only the ones that are valid covers (every edge has an endpoint inside the subset), and returns the smallest valid one found. Like brute force Max-Cut in Lab 1, this is guaranteed correct and useless past a few dozen vertices.


In [ ]:
def mvc_brute_force(G):
    '''Exact Minimum Vertex Cover. Checks every subset of vertices.'''
    nodes = list(G.nodes())
    n = len(nodes)
    edges = list(G.edges())

    for size in range(n + 1):
        for subset in itertools.combinations(nodes, size):
            subset_set = set(subset)
            if all(u in subset_set or v in subset_set for u, v in edges):
                return subset_set, size

    return set(nodes), n


for name, G in [("A", GA), ("B", GB), ("C", GC_)]:
    cover, size = mvc_brute_force(G)
    print(f"Graph {name}: brute force cover = {sorted(cover)}, size = {size}")


### 1.3.2 A fast classical fallback: matching based 2-approximation

Instead of searching all subsets, a much faster classical approach repeatedly picks any edge that is not yet covered and adds **both** of its endpoints to the cover, then discards every edge that now has a covered endpoint, and repeats until no edges remain.

This is not optimal in general, but it comes with a guarantee that a brute force search does not give you for free: the cover it returns is never more than twice the size of the true minimum. That guarantee is the whole reason this heuristic is trusted in practice instead of just being "a guess that is probably fine."


In [ ]:
def mvc_matching_approx(G):
    '''2-approximation for Minimum Vertex Cover.
    Repeatedly takes an uncovered edge and adds both endpoints.'''
    cover = set()
    remaining_edges = set(G.edges())

    while remaining_edges:
        u, v = remaining_edges.pop()
        cover.add(u)
        cover.add(v)
        remaining_edges = {(a, b) for a, b in remaining_edges if a not in cover and b not in cover}

    return cover


for name, G in [("A", GA), ("B", GB), ("C", GC_)]:
    cover = mvc_matching_approx(G)
    print(f"Graph {name}: approximate cover = {sorted(cover)}, size = {len(cover)}")


> ### Explore Further: Classical Vertex Cover
>
> - **[NetworkX: min_weighted_vertex_cover](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.approximation.vertex_cover.min_weighted_vertex_cover.html)**
>   NetworkX's built in 2-approximation, implemented the same way as the function above but generalized to weighted vertices.
>
> - **[GeeksforGeeks: Vertex Cover Problem, Set 2 (Approximate Algorithm)](https://www.geeksforgeeks.org/dsa/introduction-and-approximate-solution-for-vertex-cover-problem/)**
>   Walks through the same matching based approximation with additional worked examples.
>
> - **[Konig's theorem, bipartite graphs](https://en.wikipedia.org/wiki/K%C5%91nig%27s_theorem_(graph_theory))**
>   In bipartite graphs specifically, minimum vertex cover can be solved exactly and efficiently, because it equals the size of a maximum matching. This exact polynomial case does not extend to general graphs, which is exactly why the problem is hard in general.


## 1.4 Where classical algorithms struggle

Minimum Vertex Cover is NP-hard on general graphs. That means brute force is exponential, and the fast matching heuristic only promises to stay within a factor of 2 of optimal, not that it will find the optimal answer. Let's measure both of these effects directly instead of just stating them.


### 1.4.1 Brute force timing

Same experiment style as Lab 1: grow the graph and watch brute force runtime.

**In-class task 1.** Run the cell below as is, then change `node_range` to reach a couple of nodes further (for example `range(4, 25, 2)`) and re-run. Watch how much longer the last couple of points take compared to the first few.


In [ ]:
node_range = range(4, 21, 2)
mvc_times = []

for n in node_range:
    G_test = nx.erdos_renyi_graph(n=n, p=0.3, seed=RNG_SEED)
    start = time.time()
    mvc_brute_force(G_test)
    elapsed = time.time() - start
    mvc_times.append(elapsed)
    print(f"n = {n:2d} nodes   time = {elapsed:8.4f} s")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(node_range), mvc_times, marker="o", color="#1f4e79")
ax.set_yscale("log")
ax.set_xlabel("Number of vertices (n)")
ax.set_ylabel("Runtime in seconds (log scale)")
ax.set_title("Brute force Minimum Vertex Cover: runtime grows exponentially with n")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


### 1.4.2 Where the approximation ratio actually bites

The 2-approximation is fast at any size, but "never worse than twice optimal" is a ceiling, not a promise of being close. Let's build a graph where the gap between the approximate cover and the true minimum is visible, and compare it against brute force while brute force is still possible to run.

**In-class task 2.** Change `p=0.18` below to a higher value like `0.9` and re-run both cells. Does the approximation gap get bigger or smaller on a denser graph?


In [ ]:
G_gap = nx.erdos_renyi_graph(n=16, p=0.18, seed=RNG_SEED + 3)

start = time.time()
exact_cover, exact_size = mvc_brute_force(G_gap)
exact_time = time.time() - start

start = time.time()
approx_cover = mvc_matching_approx(G_gap)
approx_time = time.time() - start

print(f"Exact minimum cover size:      {exact_size}   (time: {exact_time:.4f} s)")
print(f"Approximate cover size:        {len(approx_cover)}   (time: {approx_time:.6f} s)")
print(f"Ratio (approx / exact):        {len(approx_cover) / exact_size:.2f}")


In [ ]:
G_gap = nx.erdos_renyi_graph(n=16, p=0.18, seed=RNG_SEED + 3)
fig, ax = plt.subplots(figsize=(6, 6))
pos = nx.spring_layout(G_gap, seed=RNG_SEED)

node_colors = ["#1E2761" if v in exact_cover else "#CADCFC" for v in G_gap.nodes()]

nx.draw(
    G_gap, pos, ax=ax, with_labels=True,
    node_color=node_colors, node_size=500,
    font_color="white", font_weight="bold",
    edge_color="#8892b0", width=1.5,
)
ax.set_title(f"n=16 random graph — exact cover size {exact_size}")
plt.tight_layout()
plt.show()

## 1.5 Formulating Minimum Vertex Cover as a QUBO

We need two things in the objective: reward small cover size, and penalize any edge that is left uncovered.

For each vertex $i$ we use a binary variable $x_i \in \{0, 1\}$, where $x_i = 1$ means vertex $i$ is included in the cover. An edge $(i, j)$ is uncovered exactly when both endpoints are excluded, that is when $x_i = 0$ and $x_j = 0$, which happens when $(1 - x_i)(1 - x_j) = 1$.

$$H = A \sum_{(i,j) \in E} (1 - x_i)(1 - x_j) + B \sum_{i \in V} x_i$$

The first term penalizes uncovered edges, the second term rewards smaller covers. As long as $A > B$, it is never worth leaving an edge uncovered just to save one vertex from the cover, so the minimum of $H$ corresponds to the minimum vertex cover. We use $B = 1$ and pick $A$ large enough relative to $B$ that violating a single edge constraint always costs more than including any number of extra vertices could save.


In [ ]:
def mvc_qubo(G, A=None, B=1.0):
    '''Builds the QUBO dictionary for Minimum Vertex Cover.
    A penalizes uncovered edges, B rewards a smaller cover. Requires A > B.'''
    if A is None:
        A = B * (G.number_of_nodes() + 1)  # comfortably larger than B

    Q = {}
    for i, j in G.edges():
        # A * (1 - x_i)(1 - x_j) = A - A*x_i - A*x_j + A*x_i*x_j
        Q[(i, i)] = Q.get((i, i), 0) - A
        Q[(j, j)] = Q.get((j, j), 0) - A
        Q[(i, j)] = Q.get((i, j), 0) + A

    for i in G.nodes():
        Q[(i, i)] = Q.get((i, i), 0) + B

    return Q


def qubo_to_cover(sample):
    return {node for node, bit in sample.items() if bit == 1}


for name, G in [("A", GA), ("B", GB), ("C", GC_)]:
    Q = mvc_qubo(G)
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q)
    result = sampler.sample(bqm, num_reads=200, num_sweeps=1000)
    cover = qubo_to_cover(result.first.sample)
    print(f"Graph {name}: QUBO cover = {sorted(cover)}, size = {len(cover)}")


### 1.5.1 QUBO vs the 2-approximation on the graph from section 1.4.2

**In-class task 3.** Run the cell below, then go back to section 1.5 and lower `A` in `mvc_qubo` to something close to `B` (say `A=0.5` or `A=1.5`) for this graph only, by calling `mvc_qubo(G_gap, A=0.5)` or `mvc_qubo(G_gap, A=1.5)` in the cell below instead. Does the QUBO solution stay a valid cover? This is the same lesson as the knapsack penalty exercise in Lab 1: an unconstrained solver has no idea a constraint even exists unless the penalty enforces it.


In [ ]:
Q_gap = mvc_qubo(G_gap)
bqm_gap = dimod.BinaryQuadraticModel.from_qubo(Q_gap)
result_gap = sampler.sample(bqm_gap, num_reads=500, num_sweeps=2000)
qubo_cover_gap = qubo_to_cover(result_gap.first.sample)

is_valid = all(u in qubo_cover_gap or v in qubo_cover_gap for u, v in G_gap.edges())

print(f"Exact minimum cover size:  {exact_size}")
print(f"Approximate cover size:    {len(approx_cover)}")
print(f"QUBO cover size:           {len(qubo_cover_gap)}")
print(f"QUBO cover is a valid cover of every edge: {is_valid}")


> ### Explore Further: Vertex Cover, QUBO & Quantum Annealing
>
> - **[Lucas, "Ising formulations of many NP problems," 2014 (arXiv:1302.5843)](https://arxiv.org/abs/1302.5843)**
>   Section on Vertex Cover gives the QUBO derivation used above, alongside a dozen other NP-hard problems formulated the same way.
>
> - **[D-Wave Examples: Vertex Cover](https://github.com/dwave-examples/vertex-cover)**
>   A worked example building this exact QUBO and sampling it, including guidance on choosing the penalty strength.
>
> - **[Qiskit Optimization: converting constrained problems to QUBO](https://qiskit-community.github.io/qiskit-optimization/tutorials/02_converters_for_quadratic_programs.html)**
>   General reference for the constraint-to-penalty conversion pattern used here and in every QUBO in this lab.


## 1.6 Minimum Vertex Cover summary

| Method | Guarantees optimum | Scales past n ~ 25 | Behavior on dense / gap-prone graphs |
|---|---|---|---|
| Brute force | Yes | No | Exact but exponential regardless of graph shape |
| Matching based 2-approximation | No | Yes | Always within 2x optimal, but that gap can be visible in practice |
| Simulated annealing (QUBO) | No, but consistently close when penalty is tuned correctly | Yes | Matches the approximation or beats it, but only stays valid if the penalty A is large enough |

The lesson to carry into Graph Coloring: writing a constraint as a penalty term does not remove the constraint, it just moves the responsibility for respecting it from the algorithm to the person choosing the penalty strength.


---
# Part 2: Graph Coloring

Given a graph $G = (V, E)$ and a number of colors $k$, graph coloring asks whether every vertex can be assigned one of the $k$ colors so that no two adjacent vertices share a color. The smallest $k$ for which this is possible is called the chromatic number of the graph, written $\chi(G)$.


## 2.1 Building intuition step by step

Let's work through this slowly on two small graphs before defining anything formally.

**Step 1: a triangle.** Three vertices, every pair connected. Try to color it with 2 colors: color vertex 0 red, vertex 1 must differ from 0 so it becomes blue, vertex 2 is adjacent to both 0 and 1, so it cannot be red or blue. Two colors is not enough. With 3 colors it works immediately: red, blue, green. So $\chi(\text{triangle}) = 3$.

**Step 2: a 4-cycle.** Four vertices in a ring, 0-1-2-3-0. Try 2 colors: 0 red, 1 blue (adjacent to 0), 2 red (adjacent to 1, and not adjacent to 0), 3 blue (adjacent to 2 and to 0, and blue is not used by either). This works: red, blue, red, blue. So $\chi(\text{4-cycle}) = 2$.

The difference between these two small graphs is the entire reason chromatic number is interesting: it is not just "the number of vertices" or "the number of edges," it depends on the specific structure of who is adjacent to whom.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

G_tri = nx.complete_graph(3)
tri_colors = {0: "#d62728", 1: "#1f77b4", 2: "#2ca02c"}
nx.draw(G_tri, nx.circular_layout(G_tri), ax=axes[0], with_labels=True,
        node_color=[tri_colors[n] for n in G_tri.nodes()], font_color="white", node_size=600)
axes[0].set_title("Triangle: needs 3 colors")

G_cyc = nx.cycle_graph(4)
cyc_colors = {0: "#d62728", 1: "#1f77b4", 2: "#d62728", 3: "#1f77b4"}
nx.draw(G_cyc, nx.circular_layout(G_cyc), ax=axes[1], with_labels=True,
        node_color=[cyc_colors[n] for n in G_cyc.nodes()], font_color="white", node_size=600)
axes[1].set_title("4-cycle: 2 colors suffice")

plt.tight_layout()
plt.show()


## 2.2 Why anyone outside a classroom cares about this

- **Compiler register allocation.** Variables that are alive at the same time become adjacent vertices, colors become CPU registers, a valid coloring is an assignment of variables to registers with no conflicts.
- **Exam and course timetabling.** Courses with a shared student become adjacent vertices, colors become time slots, a valid coloring is a timetable with no student double booked.
- **Wireless frequency assignment.** Transmitters that would interfere with each other become adjacent vertices, colors become frequency channels, a valid coloring is an assignment with no interference.
- **Map coloring.** Regions sharing a border become adjacent vertices, this is the classical motivating example and the source of the famous four color theorem for planar maps.
- **Sudoku.** Every row, column, and 3x3 box constraint can be expressed as a graph coloring instance with 9 colors, this is a common example used to demonstrate coloring solvers because the answer is easy to check by eye.


## 2.3 Classical approaches

We will look at exact brute force and the greedy heuristic taught in most algorithms courses.


### 2.3.1 Brute force k-coloring check

For $n$ vertices and $k$ colors there are $k^n$ possible colorings. Brute force tries every one and checks whether any adjacent pair shares a color, stopping at the first valid coloring it finds.


In [ ]:
def is_valid_coloring(G, coloring):
    return all(coloring[u] != coloring[v] for u, v in G.edges())


def graph_coloring_brute_force(G, k):
    '''Checks every k^n coloring and returns the first valid one, if any exists.'''
    nodes = list(G.nodes())
    n = len(nodes)

    for combo in itertools.product(range(k), repeat=n):
        coloring = dict(zip(nodes, combo))
        if is_valid_coloring(G, coloring):
            return coloring

    return None


for k in [2, 3]:
    result = graph_coloring_brute_force(G_tri, k)
    status = "found a valid coloring" if result else "no valid coloring exists"
    print(f"Triangle with k = {k} colors: {status}")


### 2.3.2 A fast classical fallback: greedy coloring

Greedy coloring visits vertices one at a time, in some order, and assigns each vertex the lowest numbered color not already used by its already colored neighbors. It never backtracks or reconsiders. It always produces a valid coloring, but not necessarily with the fewest colors possible, and the number of colors it ends up using depends heavily on the order the vertices are visited in.


In [ ]:
def greedy_coloring(G, order=None):
    '''Greedy coloring. Uses the lowest available color for each vertex in turn.'''
    if order is None:
        order = list(G.nodes())

    coloring = {}
    for node in order:
        used = {coloring[nb] for nb in G.neighbors(node) if nb in coloring}
        color = 0
        while color in used:
            color += 1
        coloring[node] = color

    return coloring


coloring_default = greedy_coloring(G_tri)
print(f"Greedy coloring (default order): {coloring_default}")
print(f"Colors used: {len(set(coloring_default.values()))}")


> ### Explore Further: Classical Graph Coloring
>
> - **[NetworkX: greedy_color](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.coloring.greedy_color.html)**
>   NetworkX's built in greedy coloring, with several vertex ordering strategies (`largest_first`, `smallest_last`, `DSATUR`, and others) you can swap in directly.
>
> - **[DSATUR algorithm](https://en.wikipedia.org/wiki/DSatur)**
>   A smarter ordering strategy that picks the next vertex by "degree of saturation" instead of a fixed order, which tends to use noticeably fewer colors than a naive greedy pass.
>
> - **[GeeksforGeeks: Graph Coloring, Backtracking](https://www.geeksforgeeks.org/dsa/graph-coloring-applications/)**
>   An exact backtracking solver that is faster than brute force in practice but still exponential in the worst case.


## 2.4 Where classical algorithms struggle

Determining the chromatic number exactly is NP-hard, and even just deciding whether a graph can be colored with 3 colors is NP-complete. Brute force pays for this directly through runtime, and greedy pays for it by silently using more colors than necessary depending on vertex order, without ever telling you it happened.


### 2.4.1 Brute force timing

**In-class task 4.** Run the cell below, then change `k=3` to `k=2` and re-run. Notice how much the runtime changes even though `n` did not, since a smaller $k$ shrinks the $k^n$ search space directly.


In [ ]:
G_test = nx.erdos_renyi_graph(n=12, p=0.4, seed=RNG_SEED)
print("Nodes:", list(G_test.nodes()))
print("Edges:", list(G_test.edges()))
print("Number of nodes:", G_test.number_of_nodes())
print("Number of edges:", G_test.number_of_edges())

pos = nx.spring_layout(G_test, seed=RNG_SEED)
plt.figure(figsize=(5, 5))
nx.draw(G_test, pos, with_labels=True, node_color="#1f4e79", font_color="white", node_size=550)
plt.title("G_test")
plt.show()

In [ ]:
node_counts_gc = list(range(4, 14, 2))
k_fixed = 3
gc_times = []

for n in node_counts_gc:
    G_test = nx.erdos_renyi_graph(n=n, p=0.4, seed=RNG_SEED)
    start = time.time()
    graph_coloring_brute_force(G_test, k_fixed)
    elapsed = time.time() - start
    gc_times.append(elapsed)
    print(f"n = {n:2d} nodes, k = {k_fixed}   time = {elapsed:8.4f} s")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(node_counts_gc, gc_times, marker="o", color="#1f4e79")
ax.set_yscale("log")
ax.set_xlabel("Number of vertices (n)")
ax.set_ylabel("Runtime in seconds (log scale)")
ax.set_title(f"Brute force {k_fixed}-coloring check: runtime grows exponentially with n")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


### 2.4.2 Greedy's hidden weak spot: vertex order

Unlike the runtime wall above, greedy's failure mode does not show up as an error or a slow cell, it shows up as a worse answer with no warning attached. Let's build a graph where two different visiting orders lead to a different number of colors used, on the exact same graph.

**In-class task 5.** Run the cell below, then try a third ordering of your own, for example `sorted(G_order.nodes(), reverse=True)`, and see if you can beat both results already shown.


In [ ]:
G_order = nx.erdos_renyi_graph(n=10, p=0.4, seed=RNG_SEED + 5)
print("Nodes:", list(G_order.nodes()))
print("Edges:", list(G_order.edges()))
print("Number of nodes:", G_order.number_of_nodes())
print("Number of edges:", G_order.number_of_edges())

pos = nx.spring_layout(G_order, seed=RNG_SEED)
plt.figure(figsize=(5, 5))
nx.draw(G_order, pos, with_labels=True, node_color="#1f4e79", font_color="white", node_size=550)
plt.title("G_order")
plt.show()

In [ ]:
G_order = nx.erdos_renyi_graph(n=10, p=0.4, seed=RNG_SEED + 5)

order_a = list(G_order.nodes())
order_b = sorted(G_order.nodes(), key=lambda v: G_order.degree(v), reverse=True)

coloring_a = greedy_coloring(G_order, order=order_a)
coloring_b = greedy_coloring(G_order, order=order_b)

print(f"Greedy, natural order:            {len(set(coloring_a.values()))} colors used")
print(f"Greedy, highest degree first:      {len(set(coloring_b.values()))} colors used")
#print(f"Greedy, reverse node order:      {len(set(coloring_c.values()))} colors used")


## 2.5 Formulating Graph Coloring as a QUBO

Fix the number of colors at $k$. For every vertex $v$ and color $c$ we use a binary variable $x_{v,c} \in \{0, 1\}$, where $x_{v,c} = 1$ means vertex $v$ is assigned color $c$.

Two things need to be enforced through penalties, since a QUBO has no native concept of "exactly one" or "not equal":

**Each vertex gets exactly one color.**
$$H_{\text{one-color}} = A \sum_{v \in V} \left(1 - \sum_{c=1}^{k} x_{v,c}\right)^2$$

**Adjacent vertices never share a color.**
$$H_{\text{adjacent}} = A \sum_{(u,v) \in E} \sum_{c=1}^{k} x_{u,c} \, x_{v,c}$$

$$H = H_{\text{one-color}} + H_{\text{adjacent}}$$

Both terms use the same penalty weight $A$ here since both are hard constraints, there is no "reward smaller size" term to balance against the way there was in Vertex Cover, because $k$ is fixed in advance rather than being minimized directly. To find the chromatic number itself, you would solve this QUBO for increasing values of $k$ until the lowest energy solution is feasible (energy 0), the same idea as the brute force check in section 2.3.1 but handed to an annealer instead of an exhaustive loop.


In [ ]:
def graph_coloring_qubo(G, k, A=2.0):
    '''Builds the QUBO dictionary for k-coloring.
    A penalizes both a vertex missing exactly one color and adjacent vertices sharing a color.'''
    Q = {}

    def var(v, c):
        return (v, c)

    def add(key, amount):
        Q[key] = Q.get(key, 0) + amount

    # one-color-per-vertex constraint: A * (1 - sum_c x_v,c)^2
    for v in G.nodes():
        for c in range(k):
            add((var(v, c), var(v, c)), -A)
        for c1 in range(k):
            for c2 in range(c1 + 1, k):
                add((var(v, c1), var(v, c2)), 2 * A)

    # adjacent-same-color constraint: A * sum_c x_u,c * x_v,c
    for u, v in G.edges():
        for c in range(k):
            add((var(u, c), var(v, c)), A)

    return Q


def qubo_to_coloring(sample, G, k):
    coloring = {}
    for v in G.nodes():
        assigned = [c for c in range(k) if sample.get((v, c), 0) == 1]
        coloring[v] = assigned[0] if len(assigned) == 1 else None
    return coloring


Q_tri = graph_coloring_qubo(G_tri, k=3)
bqm_tri = dimod.BinaryQuadraticModel.from_qubo(Q_tri)
result_tri = sampler.sample(bqm_tri, num_reads=200, num_sweeps=1000)
coloring_tri = qubo_to_coloring(result_tri.first.sample, G_tri, k=3)

print(f"QUBO coloring of the triangle with k=3: {coloring_tri}")
print(f"Valid: {None not in coloring_tri.values() and is_valid_coloring(G_tri, coloring_tri)}")


### 2.5.1 Checking the QUBO against brute force on the order-sensitive graph

**In-class task 6.** Run the cell below with `k=4`, then lower it to `k=3` and `k=2` and re-run. Does the QUBO still find a valid coloring? Compare against what section 2.4.2 found with greedy on the same graph.


In [ ]:
G_order = nx.erdos_renyi_graph(n=10, p=0.4, seed=RNG_SEED + 5)
print("Nodes:", list(G_order.nodes()))
print("Edges:", list(G_order.edges()))
print("Number of nodes:", G_order.number_of_nodes())
print("Number of edges:", G_order.number_of_edges())

pos = nx.spring_layout(G_order, seed=RNG_SEED)
plt.figure(figsize=(5, 5))
nx.draw(G_order, pos, with_labels=True, node_color="#1f4e79", font_color="white", node_size=550)
plt.title("G_order")
plt.show()

In [ ]:
k_check = 4
Q_order = graph_coloring_qubo(G_order, k=k_check, A=2.0)
bqm_order = dimod.BinaryQuadraticModel.from_qubo(Q_order)
result_order = sampler.sample(bqm_order, num_reads=500, num_sweeps=2000)
coloring_order = qubo_to_coloring(result_order.first.sample, G_order, k=k_check)

valid = None not in coloring_order.values() and is_valid_coloring(G_order, coloring_order)
colors_used = len(set(c for c in coloring_order.values() if c is not None))

print(f"QUBO with k = {k_check}: valid coloring = {valid}, distinct colors used = {colors_used}")


In [ ]:
palette = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
node_colors = [palette[coloring_order[v]] if coloring_order[v] is not None else "#cccccc" for v in G_order.nodes()]

pos = nx.spring_layout(G_order, seed=RNG_SEED)
plt.figure(figsize=(5, 5))
nx.draw(G_order, pos, with_labels=True, node_color=node_colors, font_color="white", node_size=550)
plt.title(f"QUBO coloring of G_order, k = {k_check}")
plt.show()

> ### Explore Further: Graph Coloring, QUBO & Quantum Annealing
>
> - **[Lucas, "Ising formulations of many NP problems," 2014 (arXiv:1302.5843)](https://arxiv.org/abs/1302.5843)**
>   Section on Graph Coloring gives the exact one-hot QUBO derivation used above.
>
> - **[D-Wave Examples: Map Coloring](https://github.com/dwave-examples/map-coloring)**
>   A worked graph coloring QUBO example applied to coloring a map, built on the same one-hot encoding.
>
> - **[Qiskit Optimization tutorials](https://qiskit-community.github.io/qiskit-optimization/tutorials/index.html)**
>   General reference for translating combinatorial constraints, including one-hot and mutual exclusion constraints like the ones used here, into QUBO penalty terms.


## 2.6 Graph Coloring summary

| Method | Guarantees minimum colors | Scales past n ~ 12 (for exact) | Failure mode |
|---|---|---|---|
| Brute force | Yes | No | $O(k^n)$ runtime |
| Greedy | No | Yes | Silently uses more colors than necessary, depends on vertex order |
| Simulated annealing (QUBO) | No, checks feasibility for a fixed k | Yes | Only valid if constraint penalty A is large enough, and k must be searched separately to find the true chromatic number |


---
# Part 3: Exercises

Complete these after the live session, on your own. None of them require writing a new algorithm from scratch, only changing a parameter or a line or two in cells you already ran. Re-run the relevant cells after each change and record what you observe.


**Exercise 1 (Vertex Cover, easy).** In section 1.4.1, change `p=0.3` to `p=0.5` when building `G_test` inside the timing loop, and re-run. Does brute force reach the same `n` values in a similar amount of time, or does it slow down faster? Explain in one or two sentences why the edge probability affects vertex cover search time.

**Exercise 2 (Vertex Cover, medium).** In section 1.5.1, try three different values of `A` when building `Q_gap` (for example `1.5`, `5`, and `50`), and re-run the QUBO cell each time. For each value, report the cover size and whether it is a valid cover. At what point does increasing `A` further stop changing the result?

**Exercise 3 (Graph Coloring, easy to medium).** Build a new random graph with `nx.erdos_renyi_graph(n=8, p=0.5, seed=<pick your own number>)`, then find its chromatic number by trying `graph_coloring_brute_force` with `k = 2, 3, 4, ...` until the first `k` that succeeds. Then run `greedy_coloring` on the same graph with two different vertex orderings of your choice. Report the true chromatic number and how many colors greedy used in each ordering.

**Exercise 4 (Graph G5: Vertex Cover & Graph Coloring)**

Build the fixed graph **G5** shown below, with 5 nodes and 6 edges. Then:

- Write a QUBO formulation for the **Minimum Vertex Cover (MVC)** problem on this graph and solve it using simulated annealing (`neal` sampler).  
- Write a QUBO formulation for the **Graph Coloring** problem on this graph (try with \(k=3\) colors) and solve it using simulated annealing.  
- Report your solutions: the cover set and size for MVC, and the coloring assignment for Graph Coloring and also code for brute force and greedy approaches and compare the results.  

```python
import networkx as nx
import matplotlib.pyplot as plt

# Build Graph G5
G5 = nx.Graph()
G5.add_edges_from([
    (1, 2), (2, 4), (4, 3), (3, 1),  # square
    (3, 5), (4, 5)                   # node 5 connected to 3 and 4
])

# Draw Graph G5
plt.figure(figsize=(4,4))
pos = nx.spring_layout(G5, seed=42)
nx.draw(G5, pos, with_labels=True, node_color='lightblue', node_size=800, font_size=12)
plt.title("Graph G5 (5 nodes, 6 edges)")
plt.show()


## What to submit

Save this notebook with all cells executed and their output and plots visible, along with your answers to the three exercises above (a few sentences each, plus the specific numbers you observed where asked). If you explored beyond what the exercises ask, leave a short note describing what you changed and what effect it had.
